# NBA Injury Risk — Feature Engineering
## game_logs_labeled Dataset


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

In [2]:
dest = "../data/processed"
file = 'game_logs_labeled.csv'

df = pd.read_csv(f'{dest}/{file}')

# drop the junk index
df.head()

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,...,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,is_playoff,injury_within_30_days,age
0,2016-17,203999,Nikola Jokić,1610612743,DEN,Denver Nuggets,21601225,2017-04-12,DEN @ OKC,W,...,1,5,2,4,10,29,9,0,0,22.143737
1,2016-17,201935,James Harden,1610612745,HOU,Houston Rockets,21601224,2017-04-12,HOU vs. MIN,W,...,4,1,0,3,4,27,19,0,0,27.627652
2,2016-17,1626157,Karl-Anthony Towns,1610612750,MIN,Minnesota Timberwolves,21601224,2017-04-12,MIN @ HOU,L,...,1,0,1,2,1,28,2,0,0,21.407255
3,2016-17,203932,Aaron Gordon,1610612753,ORL,Orlando Magic,21601217,2017-04-12,ORL vs. DET,W,...,1,2,0,2,4,32,17,0,0,21.571526
4,2016-17,202331,Paul George,1610612754,IND,Indiana Pacers,21601226,2017-04-12,IND vs. ATL,W,...,1,0,0,3,3,32,15,0,0,26.945927


In [3]:
#print(f"Shape: {df.shape}")
#print(f"\nDtypes:\n{df.dtypes}")
#print(f"\nDtypes:\n{df.dtypes.value_counts().to_string()}")

In [4]:
#print(f"\nDuplicates: {df.duplicated().sum()}")

In [5]:
#pd.set_option('display.max_rows', None)
#print(df.dtypes.reset_index().rename(columns={"index": "column", 0: "dtype"}))
#pd.reset_option('display.max_rows')

### Fix `GAME_DATE` dtype
##### converting `GAME_DATE` string to datetime

In [6]:
# Fix GAME_DATE dtype and sort — required before any rolling operation
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
df = df.sort_values(["PLAYER_ID", "GAME_DATE"]).reset_index(drop=True)

print(f"GAME_DATE dtype: {df['GAME_DATE'].dtype}")
print(f"Shape: {df.shape}")

GAME_DATE dtype: datetime64[us]
Shape: (216088, 35)


# Feature Engineering

All features are computed from game-level data only — no biometric or GPS data is available.
Rolling windows use `closed="left"` to exclude the current game, preventing data leakage.
Body region flags and injury history are joined using `merge_asof` for time-aware matching.

### Workload Features

| Feature | Built from | Citation |
|---|---|---|
| `rolling_7d_min` | `MIN` + `GAME_DATE` | Bourdon et al. (2017) — acute load window |
| `rolling_14d_min` | `MIN` + `GAME_DATE` | Medium-term load |
| `rolling_28d_min` | `MIN` + `GAME_DATE` | Bourdon et al. (2017) — chronic load baseline |
| `rolling_5game_min` | `MIN` (last 5 games) | Cohan et al. (2021) — 100 min in 5 games predicts muscle injury |
| `cumulative_season_min` | `MIN` + `SEASON_YEAR` | Season-long fatigue accumulation |
| `acwr` | `rolling_7d_min / rolling_28d_min` | Bourdon et al. (2017), Comas et al. (2022) — masked for first 28 days of each player-season |
| `acwr_danger_zone` | `acwr >= 1.5` | Bourdon et al. (2017) — binary flag for high-risk ratio |

### Schedule Features

| Feature | Built from | Citation |
|---|---|---|
| `rest_days` | `GAME_DATE` | Days since player's previous game |
| `is_back_to_back` | `rest_days == 1` | Comas et al. (2022) — acute load spike |
| `games_last_7d` | `GAME_DATE` | Bourdon et al. (2017) — schedule density |

### Player Features

| Feature | Built from | Citation |
|---|---|---|
| `age` | `BIRTHDATE` + `GAME_DATE` | Lu et al. (2022) — significant predictor of LEMS |
| `prior_injury_count` | `injury_data_cleaned.csv` | Lu et al. (2022) — OR 21.0, strongest single predictor |

### Recent Injury History (56-day window)

| Feature | Body Region | Citation |
|---|---|---|
| `recent_hamstring_injury` | Hamstring | Lu et al. (2022) — OR 2.39 |
| `recent_quadriceps_injury` | Quadriceps | Lu et al. (2022) — OR 4.31 |
| `recent_calf_injury` | Calf | Lu et al. (2022) |
| `recent_groin_injury` | Groin/adductor | Lu et al. (2022) — OR 2.90 |
| `recent_ankle_injury` | Ankle | Lu et al. (2022) — OR 2.66 |
| `recent_achilles_injury` | Achilles | Lu et al. (2022) |
| `recent_knee_injury` | Knee | Lu et al. (2022) |
| `recent_back_injury` | Back | Bourdon et al. (2017) |
| `recent_hip_injury` | Hip | Lu et al. (2022) |
| `recent_foot_injury` | Foot | Lu et al. (2022) |


> **Note:** ~32% of IL placements have no body region detail ("placed on IL" with no description).
> These are retained in `prior_injury_count` but cannot contribute to region-specific flags.
> The 56-day window follows Lu et al. (2022)'s definition of "recent" injury history.

## Load df_injuries

**What it captures:** The cleaned injury dataset from notebook 02_data_cleaning.

**Why:** Body region flags and prior injury count both require a player-level injury history
joined to game logs by date. This cell prepares that dataset before any joins occur.

In [7]:
# Load cleaned injury data
df_injuries = pd.read_csv("../data/processed/injury_data_cleaned.csv")
df_injuries["Date"] = pd.to_datetime(df_injuries["Date"])

# Drop rows where PLAYER_ID couldn't be matched during fuzzy matching
df_injuries = df_injuries.dropna(subset=["PLAYER_ID"]).reset_index(drop=True)

df_injuries["PLAYER_ID"] = df_injuries["PLAYER_ID"].astype("int64")

df_injuries.head(3)

,Date,Team,Acquired,Relinquished,Notes,TEAM_ABBREVIATION,player_name_clean,matched_name,PLAYER_ID,match_score
0,2015-10-27,Bulls,NaN,Cristiano Felicio,placed on IL,CHI,Cristiano Felicio,Cristiano Felicio,1626245,100.0
1,2015-10-27,Bulls,NaN,Mike Dunleavy Jr.,placed on IL recovering from surgery on back,CHI,Mike Dunleavy,Mike Dunleavy,2399,100.0
2,2015-10-27,Cavaliers,NaN,Iman Shumpert,placed on IL recovering from surgery on right ...,CLE,Iman Shumpert,Iman Shumpert,202697,100.0


## Body Region Extraction


**Feature:** `body_region` (on `df_injuries` — intermediate column, not a model feature)

**What it captures:** Which part of the body each IL placement was for. About 68% of rows
have enough detail to be classified; the remaining ~32% ("placed on IL" with no description)
are labeled "unspecified."

**Why:** The `Notes` column contains free-text descriptions like "placed on IL with sprained
left ankle." These are structured enough to extract body region via keyword matching.
`body_region` is not used directly in modeling — it enables the recent injury flags in the
next step.

**Code:** Defines a dictionary mapping keywords to body regions — for example, both "adductor"
and "groin" map to `'groin'`, and "plantar," "toe," and "foot" all map to `'foot'`. The
`extract_region` function scans each note for the first matching keyword and returns the
corresponding region. Applied row-by-row to `df_injuries["Notes"]` using `.apply()`.


In [8]:
# =============================================================================
# BODY REGION EXTRACTION FROM INJURY NOTES
# =============================================================================
# Notes are structured enough to extract body region via keyword matching.
# ~32% of rows are "placed on IL" with no detail → labeled "unspecified".
# Remaining ~68% have consistent language enabling reliable extraction.
# Per Lu et al. (2022): recent ankle (OR 2.66), hamstring (OR 2.39),
# and groin (OR 2.9) injuries were among the top 6 predictors of LEMS.
# =============================================================================

body_region_keywords = {
    'hamstring': 'hamstring',
    'quad':      'quadriceps',
    'quadricep': 'quadriceps',
    'calf':      'calf',
    'groin':     'groin',
    'adductor':  'groin',
    'ankle':     'ankle',
    'achilles':  'achilles',
    'knee':      'knee',
    'back':      'back',
    'lumbar':    'back',
    'hip':       'hip',
    'foot':      'foot',
    'plantar':   'foot',
    'toe':       'foot',
    'shoulder':  'shoulder',
    'wrist':     'wrist',
    'elbow':     'elbow',
    'thumb':     'hand',
    'finger':    'hand',
    'hand':      'hand',
    'neck':      'neck',
    'rib':       'core',
    'oblique':   'core',
    'abdominal': 'core',
    'hernia':    'core',
    'tibia':     'leg',
    'fibula':    'leg',
    'shin':      'leg',
    'thigh':     'leg',
}

def extract_region(note):
    if pd.isna(note):
        return 'unspecified'
    note_lower = note.lower()
    for keyword, region in body_region_keywords.items():
        if keyword in note_lower:
            return region
    return 'unspecified'

df_injuries["body_region"] = df_injuries["Notes"].apply(extract_region)

In [9]:
print("Body region distribution:")
print(df_injuries["body_region"].value_counts())
print(f"\nUnspecified: {(df_injuries['body_region'] == 'unspecified').sum()} "
      f"({(df_injuries['body_region'] == 'unspecified').mean():.1%})")

Body region distribution:
body_region
unspecified    2137
knee            887
ankle           877
back            332
foot            326
hamstring       287
hip             201
groin           189
calf            180
shoulder        171
hand            121
achilles        106
wrist            94
quadriceps       56
core             53
leg              48
elbow            46
neck             41
Name: count, dtype: int64

Unspecified: 2137 (34.7%)


### Body Region Injury Flags

**Features:** `recent_hamstring_injury`, `recent_quadriceps_injury`, `recent_calf_injury`,
`recent_groin_injury`, `recent_ankle_injury`, `recent_achilles_injury`, `recent_knee_injury`,
`recent_back_injury`, `recent_hip_injury`, `recent_foot_injury`

**What they capture:** For each player-game, whether that player had an IL placement for a
specific body region within the prior 56 days (8 weeks). Each flag is binary — 1 if yes,
0 if no.

**Why:** Directly from Lu et al. (2022), who defined "recent" injury history as any IL
placement within 8 weeks of the index game. They computed the odds ratio

The clinical reasoning is that soft tissue injuries have a recovery and vulnerability window
of 6–8 weeks — a player who returns from a hamstring strain is biomechanically compensating
and at elevated re-injury risk even if they feel healthy enough to play. By using 56 days we
directly replicate Lu et al.'s operationalization and can cite their odds ratios as motivation.


In [10]:
# =============================================================================
# RECENT BODY REGION INJURY FLAGS (56-DAY WINDOW)
# =============================================================================
# For each player-game, flag whether the player had an IL placement
# for each body region within the prior 56 days (8 weeks).
# 56 days follows Lu et al. (2022) definition of "recent" injury history.
# Uses merge_asof for time-aware joining — no leakage risk.
# =============================================================================

regions = [
    'hamstring', 'quadriceps', 'calf', 'groin',
    'ankle', 'achilles', 'knee', 'back', 'hip', 'foot'
]

df_injuries["Date"] = pd.to_datetime(df_injuries["Date"])

In [11]:
df = df.sort_values("GAME_DATE").reset_index(drop=True)
df_injuries = df_injuries.sort_values("Date").reset_index(drop=True)


for region in regions:
    col_name = f"recent_{region}_injury"

    df_region = (
        df_injuries[df_injuries["body_region"] == region]
        [["PLAYER_ID", "Date"]]
        .drop_duplicates()
        .sort_values("Date")
        .reset_index(drop=True)
    )

    merged = pd.merge_asof(
        df[["PLAYER_ID", "GAME_DATE"]],
        df_region.rename(columns={"Date": "injury_date"}),
        left_on="GAME_DATE",
        right_on="injury_date",
        by="PLAYER_ID",
        direction="backward"
    )

    days_since = (merged["GAME_DATE"] - merged["injury_date"]).dt.days
    df[col_name] = (
        (days_since > 0) & (days_since <= 56)
    ).astype(int)


In [12]:
# Re-sort back to PLAYER_ID, GAME_DATE after the loop
df = df.sort_values(["PLAYER_ID", "GAME_DATE"]).reset_index(drop=True)

In [13]:
# Verify
flag_cols = [f"recent_{r}_injury" for r in regions]
print("Recent injury flag counts:")
print(df[flag_cols].sum().sort_values(ascending=False))

Recent injury flag counts:
recent_ankle_injury         10631
recent_knee_injury           7724
recent_back_injury           4041
recent_hamstring_injury      3210
recent_foot_injury           3162
recent_hip_injury            2157
recent_calf_injury           2116
recent_groin_injury          2065
recent_achilles_injury        970
recent_quadriceps_injury      507
dtype: int64


## Previous injury count

**Feature:** `prior_injury_count`

**What it captures:** For each player-game, the total number of IL placements that player had accumulated strictly before that game date, across all seasons and injury types. This is a career-to-date count, not a rolling window.

**Why:** Lu et al. (2022) found that the count of prior injuries had an odds ratio of 21.0 — by far the strongest predictor of lower-extremity muscle strain in their model. The clinical reasoning is straightforward: a player with a long injury history has demonstrated biological vulnerability — prior injuries leave scar tissue, alter movement patterns, and reduce tissue resilience, effects that compound over time. No other feature in the dataset carries this kind of predictive weight.

In [14]:
# =============================================================================
# PRIOR INJURY COUNT
# =============================================================================
# For each player-game, count the total number of IL placements
# that occurred strictly before that game date.
# Per Lu et al. (2022): previous injury count had an odds ratio of 21.0 —
# the single strongest predictor of lower extremity muscle strain in NBA athletes.
# Uses merge_asof for time-aware joining — no leakage risk.
# =============================================================================

df_injuries_sorted = (
    df_injuries[["PLAYER_ID", "Date"]]
    .dropna()
    .drop_duplicates()
    .sort_values("Date")  # sort by Date only, not PLAYER_ID first
    .reset_index(drop=True)
)

# Cumulative count of IL placements per player up to each injury date
df_injuries_sorted["prior_injury_count"] = (
    df_injuries_sorted.groupby("PLAYER_ID").cumcount()
)


In [15]:
# Time-aware merge: for each game, find the most recent prior IL placement
# and carry forward the cumulative count at that point
df = pd.merge_asof(
    df.sort_values("GAME_DATE"),  # sort by GAME_DATE only
    df_injuries_sorted.rename(columns={"Date": "injury_date"}),
    left_on="GAME_DATE",
    right_on="injury_date",
    by="PLAYER_ID",
    direction="backward"
)

# Players with no prior injuries get 0
df["prior_injury_count"] = df["prior_injury_count"].fillna(0).astype(int)
df = df.drop(columns=["injury_date"])

# Re-sort back after merge
df = df.sort_values(["PLAYER_ID", "GAME_DATE"]).reset_index(drop=True)

In [16]:
# Verify
print(f"prior_injury_count null count: {df['prior_injury_count'].isna().sum()}")
print(f"\nDistribution:")
print(df["prior_injury_count"].describe().round(2))

prior_injury_count null count: 0

Distribution:
count    216088.00
mean          2.91
std           4.26
min           0.00
25%           0.00
50%           1.00
75%           4.00
max          33.00
Name: prior_injury_count, dtype: float64


In [17]:
# print(f"\nSample:")
# print(df[["PLAYER_NAME", "GAME_DATE", "prior_injury_count"]].head(10))

## Rest days and back-to-back

**Features:** `rest_days`, `is_back_to_back`

**What they capture:** `rest_days` is how many calendar days have passed since that player's
previous game. `is_back_to_back` is 1 when `rest_days == 1` — the player played yesterday.

**Why:** Recovery time between games is a direct measure of schedule-induced fatigue. Bourdon
et al. (2017) and Comas et al. (2022) both identify short rest intervals as a key contributor
to injury risk. A back-to-back is the most extreme case — zero recovery days between games.

In [18]:
# Days since the player's previous game
# shift(1) ensures we look at the PREVIOUS game, not the current one
df["rest_days"] = (
    df.groupby("PLAYER_ID")["GAME_DATE"]
    .diff()
    .dt.days
)

# Back-to-back: played yesterday
df["is_back_to_back"] = (df["rest_days"] == 1).astype(int)

# print(df[["PLAYER_NAME", "GAME_DATE", "rest_days", "is_back_to_back"]].head(10))

## Rolling minutes features and number of games played in the last 7 days


**Features:** `rolling_7d_min`, `rolling_14d_min`, `rolling_28d_min`, `games_last_7d`

**What they capture:** Total minutes played by a player in the 7, 14, and 28 calendar days
before each game. `games_last_7d` counts how many games appeared in the same 7-day window.

**Why:** Minutes played is the best available proxy for external workload — no GPS or biometric data are available in this dataset. Rolling windows capture load accumulation across time horizons. The 7-day window measures acute load, the 14-day window provides a medium-term view, and the 28-day window represents the chronic baseline required to compute ACWR. `games_last_7d` captures schedule density independently of minutes per game. All are grounded in Bourdon et al. (2017).

In [19]:
# Set GAME_DATE as index once; rolling("7D") needs a DatetimeIndex
s = df.set_index("GAME_DATE").groupby("PLAYER_ID")["MIN"]

df["rolling_7d_min"]  = s.rolling("7D",  closed="left").sum().reset_index(level=0, drop=True).values
df["rolling_14d_min"] = s.rolling("14D", closed="left").sum().reset_index(level=0, drop=True).values
df["rolling_28d_min"] = s.rolling("28D", closed="left").sum().reset_index(level=0, drop=True).values
df["games_last_7d"]    = s.rolling("7D",  closed="left").count().reset_index(level=0, drop=True).values

In [20]:
# sample = df[df["PLAYER_ID"] == df["PLAYER_ID"].iloc[0]].head(10)
# print(df[["PLAYER_NAME", "GAME_DATE", "MIN",  "rolling_7d_min", "rolling_14d_min",  "rolling_28d_min", "games_last_7d"]].head(20))

## Rolling 5-game minute total

**Feature:** `rolling_5game_min`

**What it captures:** Total minutes played across the player's previous 5 games, regardless of the number of calendar days spanned.

**Why:** Cohan et al. (2021) found that players with more than 100 minutes in their previous 5 games had significantly higher muscle injury rates. This complements the time-based windows — a player who played 5 games in 7 days looks different in the time-based windows from one who played 5 games in 12 days, but is identical here. Together, they give the model two angles on recent load accumulation.

In [21]:
# Cohan et al. found >100 min in previous 5 games was a strong muscle injury signal.
# Count-based rolling (rolling(5)) doesn't support closed="left", so we use
# shift(1) to exclude the current game and prevent leakage.
df["rolling_5game_min"] = (
    df.groupby("PLAYER_ID")["MIN"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)


In [22]:
# print(df[["PLAYER_NAME", "GAME_DATE", "MIN", "rolling_5game_min"]].head(20))

## Cumulative season minutes

**Feature:** `cumulative_season_min`

**What it captures:** The running total of minutes a player has accumulated in the current
season up to but not including the current game.

**Why:** Rolling windows capture recent load spikes, but cumulative season minutes capture long-term fatigue that builds over an entire season. A player in game 75 with 2,800 cumulative minutes is physically depleted in ways that a 28-day rolling window cannot detect. This is particularly relevant to the end-of-season injury spike visible in the labeled data.


In [23]:
# Running total of minutes within each player-season
# shift(1) so current game is not included
df["cumulative_season_min"] = (
    df.groupby(["PLAYER_ID", "SEASON_YEAR"])["MIN"]
    .transform(lambda x: x.shift(1).cumsum())
)


In [24]:
# print(df[["PLAYER_NAME", "SEASON_YEAR", "GAME_DATE",  "MIN", "cumulative_season_min"]].head(20))

In [25]:
# Should show one SEASON_YEAR per October-to-June span, not split mid-season
df.groupby("SEASON_YEAR")["GAME_DATE"].agg(["min", "max"])

,min,max
SEASON_YEAR,,
2015-16,2015-10-27,2016-06-19
2016-17,2016-10-25,2017-06-12
2017-18,2017-10-17,2018-06-08
2018-19,2018-10-16,2019-06-13
2019-20,2019-10-22,2020-10-11
2020-21,2020-12-22,2021-07-20
2021-22,2021-10-19,2022-06-16
2022-23,2022-10-18,2023-06-12


In [26]:
# LeBron in 2015-16 — played regular season + finals
mask = (df["PLAYER_ID"] == 2544) & (df["SEASON_YEAR"] == "2015-16")
check = df[mask][["GAME_DATE", "MATCHUP", "MIN", "cumulative_season_min"]].copy()
#print(check.head(5))   # early season
#print(check.tail(10))  # end of regular season + playoffs

## ACWR

**What they capture:** `acwr` is the Acute: Chronic Workload Ratio — a single number that indicates how much a player is doing right now relative to what their body is conditioned for. `acwr_danger_zone` is a binary flag that is 1 when `acwr >= 1.5`.

**Why:** ACWR is the most cited workload metric across all four papers in this project. Bourdon et al. (2017) introduced it as a simplification of Banister's fitness-fatigue model, using rolling averages to compare recent load with a longer-term baseline. Comas et al. (2022) define acute load as the prior 7 days and chronic load as the prior 28 days, the exact windows used here. Both papers report that an ACWR of 0.8–1.3 is associated with low injury risk, while an ACWR of 1.5 or higher is linked to exponentially increasing injury risk. The key insight is that injury risk is not driven by absolute load alone. A player who consistently plays heavy minutes can tolerate high acute loads because their chronic baseline is also high. It is the mismatch between the two that predicts injury.



**Why we mask the first 28 days of each player-season:** ACWR requires a full chronic baseline to be meaningful. In the early weeks of a season, `rolling_28d_min` reflects only a partial window. A player's first game has no chronic history, and their second game has only a few days. This artificially shrinks the denominator, inflating the ratio to unrealistic levels regardless of actual workload. The threshold analysis confirms this directly:


| Threshold | ACWR >= 1.5 | ACWR >= 4.0 |
|---|---|---|
| 0 days (no mask) | 51,567 (23.9%) | 13,533 (6.3%) |
| 7 days | 43,149 (20.0%) | 5,115 (2.4%) |
| 14 days | 33,739 (15.6%) | 3,347 (1.5%) |
| 21 days | 28,307 (13.1%) | 3,347 (1.5%) |
| 28 days | 26,649 (12.3%) | 3,347 (1.5%) |


The count of ACWR >= 4.0 stabilizes at 3,347 from threshold 14 onward. These are genuine extreme-workload cases in the data, regardless of masking. But at threshold 0, 13,533 games show ACWR >= 4.0, meaning 10,186 of those flags are pure artifacts of insufficient history rather than real injury-risk signals. A model trained on unmasked ACWR would learn "early season = high risk" rather than "acute load spike relative to chronic baseline = high risk", the wrong lesson entirely.



`acwr_danger_zone` is set to 0 (not NaN) for masked rows. The first 28 days of a player's season are not flagged as dangerous because they lack the history needed to make that determination. This avoids imputation in the modeling notebook and correctly communicates "not flagged" rather than "unknown."



In [27]:
for threshold in [0, 7, 14, 21, 28]:
    season_day = (
        df.groupby(["PLAYER_ID", "SEASON_YEAR"])["GAME_DATE"]
        .transform(lambda x: (x - x.min()).dt.days)
    )
    
    acute_rate   = df["rolling_7d_min"] / 7
    chronic_rate = df["rolling_28d_min"] / 28
    
    acwr = np.where(
        season_day >= threshold,
        acute_rate / chronic_rate.replace(0, np.nan),
        np.nan
    )
    
    null_count   = np.sum(np.isnan(acwr))
    pct_null     = null_count / len(df) * 100
    danger_count = np.sum(acwr >= 1.5)
    danger_pct   = danger_count / len(df) * 100

    print(f"\nThreshold {threshold} days:")
    print(f"  NaN (insufficient history): {null_count:,} ({pct_null:.1f}%)")
    print(f"  Danger zone (ACWR >= 1.5):  {danger_count:,} ({danger_pct:.1f}%)")
    print(f"  ACWR cutoffs:")
    for cutoff in [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
        count = np.sum(acwr >= cutoff)
        pct   = count / len(df) * 100
        print(f"    >= {cutoff}: {count:,} ({pct:.1f}%)")


Threshold 0 days:
  NaN (insufficient history): 15,076 (7.0%)
  Danger zone (ACWR >= 1.5):  51,567 (23.9%)
  ACWR cutoffs:
    >= 1.5: 51,567 (23.9%)
    >= 2.0: 32,318 (15.0%)
    >= 2.5: 23,647 (10.9%)
    >= 3.0: 18,261 (8.5%)
    >= 3.5: 14,590 (6.8%)
    >= 4.0: 13,533 (6.3%)

Threshold 7 days:
  NaN (insufficient history): 23,494 (10.9%)
  Danger zone (ACWR >= 1.5):  43,149 (20.0%)
  ACWR cutoffs:
    >= 1.5: 43,149 (20.0%)
    >= 2.0: 23,900 (11.1%)
    >= 2.5: 15,229 (7.0%)
    >= 3.0: 9,843 (4.6%)
    >= 3.5: 6,172 (2.9%)
    >= 4.0: 5,115 (2.4%)

Threshold 14 days:
  NaN (insufficient history): 33,422 (15.5%)
  Danger zone (ACWR >= 1.5):  33,739 (15.6%)
  ACWR cutoffs:
    >= 1.5: 33,739 (15.6%)
    >= 2.0: 15,675 (7.3%)
    >= 2.5: 9,157 (4.2%)
    >= 3.0: 5,939 (2.7%)
    >= 3.5: 4,053 (1.9%)
    >= 4.0: 3,347 (1.5%)

Threshold 21 days:
  NaN (insufficient history): 42,670 (19.7%)
  Danger zone (ACWR >= 1.5):  28,307 (13.1%)
  ACWR cutoffs:
    >= 1.5: 28,307 (13.1%)
    >

In [28]:
# ACWR requires a full 28-day chronic baseline to be meaningful.
# Per Bourdon et al. (2017), the chronic window must reflect
# a true training baseline — early-season values do not.
# We mask the first 28 days of each player-season as NaN.

df["season_day"] = (
    df.groupby(["PLAYER_ID", "SEASON_YEAR"])["GAME_DATE"]
    .transform(lambda x: (x - x.min()).dt.days)
)

acute_rate   = df["rolling_7d_min"]  / 7
chronic_rate = df["rolling_28d_min"] / 28

df["acwr"] = np.where(
    df["season_day"] >= 28,
    acute_rate / chronic_rate.replace(0, np.nan),
    np.nan
)

df["acwr_danger_zone"] = (df["acwr"] >= 1.5).astype(int)

#print(f"ACWR null count: {df['acwr'].isna().sum()}")
#print(df[["PLAYER_NAME", "GAME_DATE", "season_day",  "rolling_7d_min", "rolling_28d_min", "acwr", "acwr_danger_zone"]].head(50))

In [29]:
print(f"\nACWR danger zone games: {df['acwr_danger_zone'].sum()}")
print(f"ACWR null count (insufficient history): {df['acwr'].isna().sum()}")


ACWR danger zone games: 26649
ACWR null count (insufficient history): 51576


## Validation Check 

In [30]:
df = df.drop(columns=["season_day"])

In [31]:
# =============================================================================
# FEATURE ENGINEERING VALIDATION
# =============================================================================

# 1. Leakage check — first game per player should have NaN for all lookback features
first_games = df.groupby("PLAYER_ID").head(1)
print("=== Leakage Check (first game per player — should all be NaN) ===")
print(first_games[[
    "PLAYER_NAME", "GAME_DATE",
    "rolling_7d_min", "rolling_28d_min",
    "rolling_5game_min", "cumulative_season_min",
    "rest_days", "prior_injury_count"
]].head(10).to_string())

# 2. Null counts — expected NaNs explained
feature_cols = [
    # workload
    "rolling_7d_min", "rolling_14d_min", "rolling_28d_min",
    "rolling_5game_min", "cumulative_season_min",
    "acwr", "acwr_danger_zone",
    
    # schedule
    "rest_days", "is_back_to_back", "games_last_7d",
    
    # player
    "age", "prior_injury_count",
    
    # recent injury flags
    "recent_hamstring_injury", "recent_quadriceps_injury",
    "recent_calf_injury", "recent_groin_injury",
    "recent_ankle_injury", "recent_achilles_injury",
    "recent_knee_injury", "recent_back_injury",
    "recent_hip_injury", "recent_foot_injury",
]

=== Leakage Check (first game per player — should all be NaN) ===
       PLAYER_NAME  GAME_DATE  rolling_7d_min  rolling_28d_min  rolling_5game_min  cumulative_season_min  rest_days  prior_injury_count
0    Kevin Garnett 2015-10-28             NaN              NaN                NaN                    NaN        NaN                   0
38     Kobe Bryant 2015-10-28             NaN              NaN                NaN                    NaN        NaN                   0
104     Tim Duncan 2015-10-28             NaN              NaN                NaN                    NaN        NaN                   0
175   Vince Carter 2015-11-02             NaN              NaN                NaN                    NaN        NaN                   0
512  Dirk Nowitzki 2015-10-28             NaN              NaN                NaN                    NaN        NaN                   0
774    Paul Pierce 2015-10-28             NaN              NaN                NaN                    NaN        NaN   

In [32]:
print("\n=== Null Counts Per Feature ===")
null_counts = df[feature_cols].isna().sum()
print(null_counts[null_counts > 0].to_string() or "No nulls in any feature")
print(f"\nExpected nulls:")
print(f"  rolling_*     — first game per player-season has no prior history")
print(f"  rest_days     — first game per player has no previous game")
print(f"  acwr          — first 28 days of each player-season masked intentionally")


=== Null Counts Per Feature ===
rolling_7d_min           15076
rolling_14d_min           7791
rolling_28d_min           5687
rolling_5game_min         1248
cumulative_season_min     4252
acwr                     51576
rest_days                 1248
games_last_7d            15076

Expected nulls:
  rolling_*     — first game per player-season has no prior history
  rest_days     — first game per player has no previous game
  acwr          — first 28 days of each player-season masked intentionally


In [33]:
# 3. Feature distributions
print("\n=== Feature Summary ===")
print(df[feature_cols].describe().round(2).to_string())


=== Feature Summary ===
       rolling_7d_min  rolling_14d_min  rolling_28d_min  rolling_5game_min  cumulative_season_min       acwr  acwr_danger_zone  rest_days  is_back_to_back  games_last_7d        age  prior_injury_count  recent_hamstring_injury  recent_quadriceps_injury  recent_calf_injury  recent_groin_injury  recent_ankle_injury  recent_achilles_injury  recent_knee_injury  recent_back_injury  recent_hip_injury  recent_foot_injury
count       201012.00        208297.00        210401.00          214840.00              211836.00  164512.00         216088.00  214840.00        216088.00      201012.00  216088.00           216088.00                216088.00                 216088.00           216088.00            216088.00            216088.00               216088.00           216088.00           216088.00          216088.00           216088.00
mean            66.66           125.61           234.01             113.27                 790.04       1.16              0.12       6.05    

In [34]:
# 4. Sanity check on ranges
print("\n=== Range Checks ===")
print(f"age range:               {df['age'].min():.1f} – {df['age'].max():.1f} years")
print(f"rest_days range:         {df['rest_days'].min():.0f} – {df['rest_days'].max():.0f} days")
print(f"acwr range:              {df['acwr'].min():.2f} – {df['acwr'].max():.2f}")
print(f"prior_injury_count max:  {df['prior_injury_count'].max()}")
print(f"acwr_danger_zone games:  {df['acwr_danger_zone'].sum():,} ({df['acwr_danger_zone'].mean():.1%})")

flag_cols = [c for c in feature_cols if c.startswith("recent_")]
print(f"\n=== Recent Injury Flag Counts ===")
print(df[flag_cols].sum().sort_values(ascending=False).to_string())


=== Range Checks ===
age range:               18.8 – 43.1 years
rest_days range:         1 – 1553 days
acwr range:              0.00 – 4.00
prior_injury_count max:  33
acwr_danger_zone games:  26,649 (12.3%)

=== Recent Injury Flag Counts ===
recent_ankle_injury         10631
recent_knee_injury           7724
recent_back_injury           4041
recent_hamstring_injury      3210
recent_foot_injury           3162
recent_hip_injury            2157
recent_calf_injury           2116
recent_groin_injury          2065
recent_achilles_injury        970
recent_quadriceps_injury      507


In [35]:
df.head()

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,...,rest_days,is_back_to_back,rolling_7d_min,rolling_14d_min,rolling_28d_min,games_last_7d,rolling_5game_min,cumulative_season_min,acwr,acwr_danger_zone
0,2015-16,708,Kevin Garnett,1610612750,MIN,Minnesota Timberwolves,21500017,2015-10-28,MIN @ LAL,W,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2015-16,708,Kevin Garnett,1610612750,MIN,Minnesota Timberwolves,21500029,2015-10-30,MIN @ DEN,W,...,2.0,0,12.886667,12.886667,12.886667,1.0,12.886667,12.886667,NaN,0
2,2015-16,708,Kevin Garnett,1610612750,MIN,Minnesota Timberwolves,21500050,2015-11-02,MIN vs. POR,L,...,3.0,0,35.203333,35.203333,35.203333,2.0,35.203333,35.203333,NaN,0
3,2015-16,708,Kevin Garnett,1610612750,MIN,Minnesota Timberwolves,21500071,2015-11-05,MIN vs. MIA,L,...,3.0,0,38.900000,51.786667,51.786667,2.0,51.786667,51.786667,NaN,0
4,2015-16,708,Kevin Garnett,1610612750,MIN,Minnesota Timberwolves,21500085,2015-11-07,MIN @ CHI,W,...,2.0,0,27.850000,63.053333,63.053333,2.0,63.053333,63.053333,NaN,0


## Save

In [36]:
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Shape: (216088, 56)
Columns: ['SEASON_YEAR', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'is_playoff', 'injury_within_30_days', 'age', 'recent_hamstring_injury', 'recent_quadriceps_injury', 'recent_calf_injury', 'recent_groin_injury', 'recent_ankle_injury', 'recent_achilles_injury', 'recent_knee_injury', 'recent_back_injury', 'recent_hip_injury', 'recent_foot_injury', 'prior_injury_count', 'rest_days', 'is_back_to_back', 'rolling_7d_min', 'rolling_14d_min', 'rolling_28d_min', 'games_last_7d', 'rolling_5game_min', 'cumulative_season_min', 'acwr', 'acwr_danger_zone']


In [37]:
df.to_csv("../data/processed/game_logs_features.csv", index=False)
print(f"Saved: game_logs_features.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Saved: game_logs_features.csv
Shape: (216088, 56)
Columns: ['SEASON_YEAR', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'is_playoff', 'injury_within_30_days', 'age', 'recent_hamstring_injury', 'recent_quadriceps_injury', 'recent_calf_injury', 'recent_groin_injury', 'recent_ankle_injury', 'recent_achilles_injury', 'recent_knee_injury', 'recent_back_injury', 'recent_hip_injury', 'recent_foot_injury', 'prior_injury_count', 'rest_days', 'is_back_to_back', 'rolling_7d_min', 'rolling_14d_min', 'rolling_28d_min', 'games_last_7d', 'rolling_5game_min', 'cumulative_season_min', 'acwr', 'acwr_danger_zone']
